Скачивание датасета и обработка для работы с 6 классами

In [ ]:
import os
import shutil
from PIL import Image

!pip install ultralytics kaggle -q
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d kushagrapandya/visdrone-dataset
!unzip -q visdrone-dataset.zip

CLASS_MAP = {
    4: 0,   # car
    5: 1,   # van
    6: 2,   # truck
    8: 3,   # awning-tricycle
    9: 4,   # bus
    10: 5,  # motor
}

def filter_and_convert(split, src_name):
    ann_in  = f"{src_name}/annotations"
    img_in  = f"{src_name}/images"
    ann_out = f"visdrone_filtered/{split}/labels"
    img_out = f"visdrone_filtered/{split}/images"

    os.makedirs(ann_out, exist_ok=True)
    os.makedirs(img_out, exist_ok=True)

    converted, skipped = 0, 0

    for ann_file in sorted(os.listdir(ann_in)):
        if not ann_file.endswith(".txt"):
            continue

        img_name = ann_file.replace(".txt", ".jpg")
        img_path = os.path.join(img_in, img_name)
        if not os.path.exists(img_path):
            continue

        img = Image.open(img_path)
        img_w, img_h = img.size

        yolo_lines = []
        with open(os.path.join(ann_in, ann_file)) as f:
            for line in f:
                parts = line.strip().split(",")
                if len(parts) < 6:
                    continue
                x, y, w, h = int(parts[0]), int(parts[1]), int(parts[2]), int(parts[3])
                cat = int(parts[5])

                if cat not in CLASS_MAP or w == 0 or h == 0:
                    continue

                cx = (x + w / 2) / img_w
                cy = (y + h / 2) / img_h
                nw = w / img_w
                nh = h / img_h

                yolo_lines.append(
                    f"{CLASS_MAP[cat]} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}"
                )

        if yolo_lines:
            with open(os.path.join(ann_out, ann_file), "w") as f:
                f.write("\n".join(yolo_lines))
            shutil.copy(img_path, img_out)
            converted += 1
        else:
            skipped += 1

filter_and_convert("train", "VisDrone2019-DET-train/VisDrone2019-DET-train")
filter_and_convert("val",   "VisDrone2019-DET-val/VisDrone2019-DET-val")

yaml_content = """
path: /content/visdrone_filtered
train: train/images
val: val/images

nc: 6
names:
  0: car
  1: van
  2: truck
  3: awning-tricycle
  4: bus
  5: motor
"""

with open("VisDrone_filtered.yaml", "w") as f:
    f.write(yaml_content)


train: ✅ сохранено 6288, ⏭ пропущено 183
val: ✅ сохранено 543, ⏭ пропущено 5


Обучение модели yolov8m

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8m.pt")

model.train(
    data="VisDrone_filtered.yaml",
    epochs=50,
    imgsz=640,
    batch=-1,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.3,
    flipud=0.5,
    fliplr=0.5,
    degrees=15,
    scale=0.7,
    translate=0.2,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    dropout=0.1,
    label_smoothing=0.1,
    patience=15,
    workers=4,
    cache=True,
    plots=True,
    save_period=5,
    cos_lr=True,
)

New https://pypi.org/project/ultralytics/8.4.65 available 😃 Update with 'pip install -U ultralytics'
WARNING ⚠️ 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.4.64 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=VisDrone_filtered.yaml, degrees=15, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.1, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train,